# Cálculo Estocástico para Finanzas
## Compilación de Laboratorios 1–5 + Proyecto RA1000 + Tarea

**Universidad Panamericana — Facultad de Ciencias Empresariales**  
**Profesor:** Dr. Julio César Galindo  
**Equipo:** Juan Pablo Arciniega · Santiago Sabat · Mauricio Olivares  
**Fecha:** 27 de mayo de 2026

---

### Estructura del documento

| Sección | Tema |
|---------|------|
| **RA1000** | Proyecto 9: Factores y Estilos de Inversión (AAPL 2019–2024) + datos Excel |
| **Tarea** | Procesos de Poisson, Cadenas de Markov y Martingalas Discretas |
| **Lab 1** | Estimación de un Movimiento Browniano Geométrico |
| **Lab 2** | Volatilidad Realizada y Ventanas Móviles |
| **Lab 3** | Valuación Monte Carlo de una Opción Europea |
| **Lab 4** | Opción Americana con Longstaff–Schwartz |
| **Lab 5** | Opción Parisina y Reloj de Barrera |

In [ ]:
# ── Instalación y configuración global ──────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'yfinance', '-q'])

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import norm
import yfinance as yf
import datetime

# Estilo global
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.figsize': (13, 6), 'figure.dpi': 110,
                     'axes.titlesize': 14, 'axes.labelsize': 12})
np.random.seed(42)
print('Librerías cargadas correctamente.')

---
# RA1000 — Proyecto 9: Factores y Estilos de Inversión | Apple Inc. (AAPL) 2019–2024

> **Pregunta central:** ¿Puede describirse el comportamiento de un activo mediante exposiciones a estilos o factores empíricos?

## 1. Marco Teórico

### CAPM
$$E(R_i) = R_f + \beta_i\,[E(R_m) - R_f]$$

### Fama-French 3 Factores
$$R_i - R_f = \alpha + b_1\cdot(\text{Mkt-RF}) + b_2\cdot\text{SMB} + b_3\cdot\text{HML} + \varepsilon_i$$

- **SMB** (Small Minus Big): prima de tamaño  
- **HML** (High Minus Low): prima de valor (precio/libro)

## 2. Datos y Metodología

- **Activo:** AAPL (NASDAQ) | **Periodo:** Feb 2019 – Dic 2024 (71 obs. mensuales)  
- **Fuentes:** Yahoo Finance (AAPL) + Kenneth French Data Library (factores)  
- **Estimación:** OLS con errores robustos HC3 (White, 1980)

In [ ]:
# ── RA1000: Estadísticas descriptivas (datos del Excel RA1000_Excel_Calculations.xlsx) ──

# Los valores provienen de la hoja 'Datos Mensuales' del Excel entregado.
stats = pd.DataFrame({
    'Variable':  ['AAPL', 'Mkt-RF', 'SMB', 'HML', 'RF'],
    'Media (%)': [2.17,    1.37,   -0.24,   0.13,  0.20],
    'Desv. Est.':[8.45,    5.20,    2.70,   3.84,  0.18],
    'Min (%)':  [-14.70, -13.37,  -7.10,  -9.89,  0.01],
    'Max (%)':  [21.34,   13.66,   7.47,   9.81,  0.47],
    'Sharpe':   [0.257,   0.280,    None,   None,  None]
}).set_index('Variable')

print('Tabla 1. Estadísticas descriptivas — Retornos mensuales (%)')
print('=' * 65)
print(stats.to_string())
print(f'\nCorrelación AAPL-Mercado: 0.874')

In [ ]:
# ── RA1000: Resultados CAPM y Fama-French (de las hojas 'CAPM' y 'Fama-French 3F') ──

capm = {
    'Coef':    {'α (Alfa)': 0.0003,  'β (Mkt)': 1.4192},
    'Err Std': {'α (Alfa)': 0.0051,  'β (Mkt)': 0.0950},
    't-stat':  {'α (Alfa)': 0.0592,  'β (Mkt)': 14.938},
    'p-valor': {'α (Alfa)': 0.9530,  'β (Mkt)': 0.0000},
}
ff3 = {
    'Coef':    {'α':  -0.0004, 'b1 (Mkt)': 1.4594, 'b2 (SMB)': -0.0022, 'b3 (HML)': -0.0029},
    'Err Std': {'α':   0.0049, 'b1 (Mkt)': 0.0952, 'b2 (SMB)':  0.0023, 'b3 (HML)':  0.0015},
    't-stat':  {'α':  -0.165,  'b1 (Mkt)': 12.673, 'b2 (SMB)': -0.498,  'b3 (HML)': -1.865},
    'p-valor': {'α':   0.869,  'b1 (Mkt)':  0.000, 'b2 (SMB)':  0.619,  'b3 (HML)':  0.062},
}

metrics = pd.DataFrame({
    'Métrica':       ['R²', 'R² Ajustado', 'AIC', 'BIC', 'F incremental'],
    'CAPM':          [0.7638, 0.7565, -450.40, -445.88, '—'],
    'FF3':           [0.7815, 0.7878, -457.10, -448.05, 'F=4.155, p=0.020 ✓'],
}).set_index('Métrica')

print('Tabla 2. Resultados CAPM')
print(pd.DataFrame(capm).to_string())
print('\nTabla 3. Resultados Fama-French 3F')
print(pd.DataFrame(ff3).to_string())
print('\nTabla 4. Comparación de modelos')
print(metrics.to_string())

In [ ]:
# ── RA1000: Visualización comparativa de modelos ──────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel izquierdo: R² comparativo
models = ['CAPM', 'Fama-French 3F']
r2s    = [76.38, 78.15]
colors = ['#2c7bb6', '#1a9641']
bars = axes[0].bar(models, r2s, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
axes[0].set_ylim(74, 80)
axes[0].set_ylabel('R² (%)')
axes[0].set_title('Poder explicativo (R²)')
for bar, val in zip(bars, r2s):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f'{val:.2f}%', ha='center', va='bottom', fontweight='bold')

# Panel derecho: Coeficientes FF3 con barras de error
coefs  = [-0.0004, 1.4594, -0.0022, -0.0029]
errs   = [ 0.0049, 0.0952,  0.0023,  0.0015]
labels = ['α', 'b₁ (Mkt)', 'b₂ (SMB)', 'b₃ (HML)']
cols2  = ['gray' if abs(c/e) < 2 else '#1a9641' for c, e in zip(coefs, errs)]
axes[1].barh(labels, coefs, xerr=errs, color=cols2, edgecolor='white',
             height=0.5, capsize=4)
axes[1].axvline(0, color='black', lw=0.8, ls='--')
axes[1].set_title('Coeficientes Fama-French 3F (HC3)')
axes[1].set_xlabel('Estimado')

fig.suptitle('RA1000 — Proyecto 9: Análisis de Factores AAPL 2019–2024',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nPerfil factorial de AAPL:')
print('  • Alta sensibilidad al mercado (β ≈ 1.46, p < 0.01)')
print('  • Large-cap  (b₂ < 0, no significativo)')
print('  • Growth     (b₃ < 0, marginalmente significativo p ≈ 0.06)')
print('  • Alfa no significativo → sin retornos anormales robustos')

---
# Tarea — Cálculo Estocástico: Procesos de Poisson, Cadenas de Markov y Martingalas Discretas

*Autor: Juan Pablo Arciniega — Universidad Panamericana — Verano 2026*

Esta sección presenta soluciones selectas de los tres temas principales de la tarea.
La resolución completa y detallada se encuentra en el PDF `Tarea_Calculo_Estocastico.pdf`.

## 1. Proceso de Poisson — Resultados clave

In [ ]:
# ── Tarea: Proceso de Poisson — Cálculos numéricos ───────────────────────
import math

lam = 10  # órdenes/hora

# Ejercicio 1.1(a): P(N=0) en 30 min
mu_30min = lam * 0.5
p_zero   = math.exp(-mu_30min)

# Ejercicio 1.1(b): P(N1=3, N2=7) en intervalos disjuntos
p_3 = math.exp(-mu_30min) * mu_30min**3 / math.factorial(3)
p_7 = math.exp(-mu_30min) * mu_30min**7 / math.factorial(7)

# Ejercicio 1.1(c): Jornada de 8 horas
E_jornada = lam * 8
std_jornada = math.sqrt(E_jornada)

# Ejercicio 1.9(b): Intensidad intradiaria integrada
from scipy.integrate import quad
integrand = lambda t: 20 * (1 + 0.5 * math.cos(math.pi * (t - 9) / 8))
Lambda_total, _ = quad(integrand, 9, 17)

print('Ejercicio 1.1 — Llegadas de órdenes (λ=10 ord/hora)')
print(f'  (a) P(N=0 en 30 min) = e^(-5) ≈ {p_zero:.6f}  ({p_zero*100:.4f}%)')
print(f'  (b) P(N₁=3)·P(N₂=7) ≈ {p_3*p_7:.6f}  ({p_3*p_7*100:.4f}%)')
print(f'  (c) E[N(8h)] = {E_jornada},  σ = √{E_jornada} ≈ {std_jornada:.4f}')
print(f'\nEjercicio 1.9 — Intensidad intradiaria integrada:')
print(f'  Λ(17) - Λ(9) = {Lambda_total:.2f} órdenes esperados en la jornada')

In [ ]:
# ── Tarea: Cadenas de Markov — Distribución estacionaria ─────────────────

P_markov = np.array([[0.0, 0.4, 0.6],
                     [0.3, 0.0, 0.7],
                     [0.5, 0.5, 0.0]])

# Resolver πP = π con Σπᵢ = 1  →  (Pᵀ - I)π = 0
A = (P_markov.T - np.eye(3))
A = np.vstack([A, np.ones(3)])
b = np.zeros(4); b[-1] = 1
pi_stat, _, _, _ = np.linalg.lstsq(A, b, rcond=None)

# Ej 2.6: Vida esperada libre de default para rating model
# Submatriz Q (transiente): estados {AAA, BBB, CCC}
Q = np.array([[0.90, 0.08, 0.01],
              [0.05, 0.85, 0.08],
              [0.01, 0.06, 0.82]])
I_Q = np.eye(3) - Q
ones = np.ones(3)
m_vec = np.linalg.solve(I_Q, ones)

print('Ejercicio 2.1 — Distribución estacionaria (3 estados):')
for i, p in enumerate(pi_stat, 1):
    print(f'  π_{i} ≈ {p:.4f}  ({p*100:.2f}%)')

print('\nEjercicio 2.6 — Vida esperada libre de default (años):')
ratings = ['AAA', 'BBB', 'CCC']
for r, m in zip(ratings, m_vec):
    print(f'  m_{r} ≈ {m:.1f} años')

In [ ]:
# ── Tarea: Martingalas — Criterio de Kelly ────────────────────────────────
from scipy.optimize import brentq

u, d, p = 1.2, 0.8, 0.6

G = lambda f: p*np.log(1+0.2*f) + (1-p)*np.log(1-0.2*f)

# Máximo de G(f): G'(f)=0  →  0.6·0.2/(1+0.2f) - 0.4·0.2/(1-0.2f)=0
dG = lambda f: p*0.2/(1+0.2*f) - (1-p)*0.2/(1-0.2*f)
f_star = brentq(dG, 0.01, 4.9)

# fmax: G(fmax)=0
f_max = brentq(G, f_star+0.01, 4.99)

f_range = np.linspace(0, 5, 500)
G_vals  = [G(f) if 0.2*f < 1 else np.nan for f in f_range]

plt.figure(figsize=(9, 5))
plt.plot(f_range, G_vals, color='steelblue', lw=2)
plt.axvline(f_star, color='green', ls='--', label=f'f* = {f_star:.2f} (Kelly óptimo)')
plt.axvline(f_max,  color='red',   ls='--', label=f'f_max ≈ {f_max:.2f} (ruina c.s.)')
plt.axhline(0, color='black', lw=0.8)
plt.fill_between(f_range, G_vals, 0,
                 where=[g is not None and g > 0 for g in G_vals],
                 alpha=0.15, color='green')
plt.title('Criterio de Kelly — G(f) = tasa de crecimiento logarítmica', fontweight='bold')
plt.xlabel('Fracción invertida f'); plt.ylabel('G(f)')
plt.legend(); plt.grid(True, ls='--', alpha=0.6)
plt.tight_layout(); plt.show()

print(f'Ej 3.5 — Criterio de Kelly (u={u}, d={d}, p={p})')
print(f'  f* = {f_star:.4f}  → invertir el {f_star*100:.0f}% del capital')
print(f'  f_max ≈ {f_max:.4f} → para f > f_max, W_n → 0 c.s. (ruina asintótica)')

---
# Laboratorio 1 — Estimación de un Movimiento Browniano Geométrico con Datos de una Acción

**Objetivo:** Calibrar los parámetros del GBM a partir de datos históricos de AAPL,  
simular 100 trayectorias y comparar con la trayectoria real.

$$S_{t+\Delta t} = S_t\exp\!\left[\left(\mu_b - \frac{\sigma_b^2}{2}\right)\Delta t + \sigma_b\sqrt{\Delta t}\,Z_t\right],\quad Z_t\sim N(0,1)$$

**Parámetros anualizados:**
$$\hat{\mu}_b = 252\,\bar{r} + \tfrac{1}{2}\cdot 252\,\hat{s}^2 \qquad \hat{\sigma}_b = \sqrt{252}\,\hat{s}$$

In [ ]:
# ── Laboratorio 1: Paso 1-4 — Descarga y calibración ─────────────────────

ticker_l1 = 'AAPL'
df_l1 = yf.download(ticker_l1, start='2020-01-01', end='2023-12-31', progress=False)
if isinstance(df_l1.columns, pd.MultiIndex):
    df_l1.columns = df_l1.columns.droplevel(1)
prices_l1 = df_l1['Close'].dropna()

log_ret_l1 = np.log(prices_l1).diff().dropna()
mb   = log_ret_l1.mean()
sb   = log_ret_l1.std()
N_td = 252
mu_b    = N_td * mb + 0.5 * N_td * sb**2
sigma_b = np.sqrt(N_td) * sb

print(f'Ticker:  {ticker_l1} | Periodo: 2020-01-01 a 2023-12-31')
print(f'Observaciones: {len(prices_l1)} días')
print(f'\nParámetros diarios:  m̂_b = {mb:.8f}  |  ŝ_b = {sb:.8f}')
print(f'Parámetros anuales:  μ_b = {mu_b:.6f}  |  σ_b = {sigma_b:.6f}')

In [ ]:
# ── Laboratorio 1: Paso 5-6 — Simulación GBM y gráfico ───────────────────

S0_l1       = float(prices_l1.iloc[-1])
num_steps_l1 = len(prices_l1)
num_sims_l1  = 100
dt_l1        = 1.0
drift_d = (mu_b - 0.5 * sigma_b**2) / N_td
diff_d  = sigma_b / np.sqrt(N_td)

np.random.seed(42)
sim_paths_l1 = np.zeros((num_steps_l1, num_sims_l1))
sim_paths_l1[0] = S0_l1
for t in range(1, num_steps_l1):
    Z = np.random.normal(0, 1, num_sims_l1)
    sim_paths_l1[t] = sim_paths_l1[t-1] * np.exp(drift_d + diff_d * Z)

p5_l1  = np.percentile(sim_paths_l1, 5,  axis=1)
p50_l1 = np.percentile(sim_paths_l1, 50, axis=1)
p95_l1 = np.percentile(sim_paths_l1, 95, axis=1)
dates_l1 = prices_l1.index

plt.figure(figsize=(14, 7))
plt.plot(dates_l1, prices_l1.values, color='royalblue', lw=2.2,
         label=f'Trayectoria Histórica Real ({ticker_l1})', zorder=5)
for i in range(min(25, num_sims_l1)):
    plt.plot(dates_l1, sim_paths_l1[:, i], color='gray', alpha=0.12, lw=0.8)
plt.plot(dates_l1, p50_l1, 'r--', lw=1.5, label='Percentil 50 (mediana sim.)')
plt.fill_between(dates_l1, p5_l1, p95_l1, color='green', alpha=0.12,
                 label='Banda 5°–95° percentil')
plt.title(f'Lab 1: {ticker_l1} — Trayectoria Histórica vs. Simulaciones GBM',
          fontweight='bold')
plt.xlabel('Fecha'); plt.ylabel('Precio (USD)')
plt.legend(fontsize=10); plt.grid(True, ls='--', alpha=0.6)
plt.tight_layout(); plt.show()

print(f'\nResumen de parámetros anualizados:')
print(f'  μ_b (media)      = {mu_b:.6f}')
print(f'  σ_b (volatilidad) = {sigma_b:.6f}')

### Conclusiones — Laboratorio 1

El GBM calibrado con datos históricos de AAPL reproduce razonablemente la tendencia general y el rango de movimiento de precios. Sin embargo, la trayectoria real frecuentemente sale de las bandas de percentiles, revelando **cuatro limitaciones estructurales** del modelo:

1. **Volatilidad constante** — el mercado exhibe *volatility clustering* (alta volat. sigue a alta volat.)  
2. **Colas pesadas** — los retornos reales tienen *fat tails*; el GBM subestima eventos extremos  
3. **Ausencia de saltos** — el GBM no modela discontinuidades por noticias imprevistas  
4. **Deriva constante** — el retorno esperado varía en el tiempo con el ciclo económico  

Modelos alternativos: saltos-difusión (Merton), volatilidad estocástica (Heston), GARCH.

---
# Laboratorio 2 — Volatilidad Realizada y Ventanas Móviles

**Objetivo:** Calcular la volatilidad realizada anualizada de SPY con ventanas de 21, 63 y 126 días,  
identificar regímenes de mercado y validar el *clustering* de volatilidad.

$$\hat{\sigma}_{w,t} = \hat{s}_{w,t}\cdot\sqrt{252}$$
donde $\hat{s}_{w,t}$ es la desviación estándar muestral de los log-retornos en la ventana de $w$ días.

In [ ]:
# ── Laboratorio 2: Paso 1-4 — Descarga y ventanas móviles ────────────────

ticker_l2 = 'SPY'
datos_l2  = yf.download(ticker_l2, start='2014-01-01', end='2024-01-01', progress=False)
if isinstance(datos_l2.columns, pd.MultiIndex):
    datos_l2.columns = datos_l2.columns.droplevel(1)
precios_l2 = datos_l2['Close'].copy()
log_r_l2   = np.log(precios_l2 / precios_l2.shift(1)).dropna()

ventanas = [21, 63, 126]
df_vol_l2 = pd.DataFrame(index=log_r_l2.index)
for v in ventanas:
    df_vol_l2[f'Vol_{v}d'] = log_r_l2.rolling(window=v).std() * np.sqrt(252)

print(f'Datos: {ticker_l2} | {len(precios_l2)} observaciones diarias (2014–2024)')
print(f'Media log-retorno diario: {log_r_l2.mean().item():.6f}')
print('\nVolatilidades promedio anualizadas:')
for v in ventanas:
    print(f'  Ventana {v:3d}d: {df_vol_l2[f"Vol_{v}d"].mean()*100:.2f}%')

In [ ]:
# ── Laboratorio 2: Paso 5 — Gráfico de volatilidad realizada ─────────────

fig, ax = plt.subplots(figsize=(15, 7))

ax.plot(df_vol_l2['Vol_21d']*100,  label='21 días (corto plazo)',
        color='#e74c3c', alpha=0.6, lw=1.2)
ax.plot(df_vol_l2['Vol_63d']*100,  label='63 días (medio plazo)',
        color='#3498db', alpha=0.8, lw=1.8)
ax.plot(df_vol_l2['Vol_126d']*100, label='126 días (largo plazo)',
        color='#2c3e50', alpha=0.9, lw=2.5)

# Sombreado de periodos turbulentos
ax.axvspan(pd.to_datetime('2020-03-01'), pd.to_datetime('2020-06-01'),
           color='red', alpha=0.12)
ax.text(pd.to_datetime('2020-04-15'), 72, 'COVID-19',
        color='darkred', fontweight='bold', ha='center', fontsize=9)
ax.axvspan(pd.to_datetime('2022-01-01'), pd.to_datetime('2022-12-31'),
           color='orange', alpha=0.10)
ax.text(pd.to_datetime('2022-07-01'), 32, 'Alzas Fed',
        color='darkorange', fontweight='bold', ha='center', fontsize=9)

ax.set_title(f'Lab 2: Volatilidad Realizada Anualizada — {ticker_l2} (S&P 500 ETF)',
             fontweight='bold')
ax.set_xlabel('Fecha'); ax.set_ylabel('Volatilidad Anualizada (%)')
ax.legend(loc='upper right', fontsize=10, frameon=True)
ax.grid(True, ls='--', alpha=0.6)
plt.tight_layout(); plt.show()

### Conclusiones — Laboratorio 2

**Regímenes de mercado:**
- *Periodos tranquilos (2014–2017):* volatilidad < 12%, mínimos ≈ 8%
- *COVID-19 (mar–may 2020):* ventana de 21d supera el **60%** anualizado
- *Alzas Fed (2022):* meseta sostenida de **25–30%** durante todo el año

**Clustering de volatilidad (Mandelbrot-Engle):**  
Los picos de volatilidad se agrupan en bloques temporales; el mercado tarda semanas en estabilizarse.

**Efecto de la dimensión de la ventana:**

| Ventana | Característica | Uso |
|---------|---------------|-----|
| 21d | Alta reactividad, exceso de ruido | Señales tácticas |
| 63d | Balance sensibilidad/suavizado | Decisiones de mediano plazo |
| 126d | Rezago apreciable, tendencia clara | Identificar cambios de régimen |

---
# Laboratorio 3 — Valuación Monte Carlo de una Opción Europea

**Objetivo:** Valuar un *call* europeo sobre SPY bajo la medida neutral al riesgo $Q$  
y contrastar con Black-Scholes.

$$\hat{V}_0^{MC} = e^{-rT}\cdot\frac{1}{N}\sum_{i=1}^N \max(S_T^{(i)}-K,\,0)$$

$$V_0^{BS} = S_0\,\Phi(d_1) - Ke^{-rT}\,\Phi(d_2)$$

In [ ]:
# ── Laboratorio 3: Calibración ────────────────────────────────────────────

ticker_l3 = 'SPY'
end_l3    = datetime.date.today()
start_l3  = end_l3 - datetime.timedelta(days=365*2)
data_l3   = yf.download(ticker_l3,
                        start=start_l3.strftime('%Y-%m-%d'),
                        end=end_l3.strftime('%Y-%m-%d'),
                        progress=False)
if isinstance(data_l3.columns, pd.MultiIndex):
    data_l3.columns = data_l3.columns.droplevel(1)

S0_l3    = float(data_l3['Close'].iloc[-1])
log_r_l3 = np.log(data_l3['Close']/data_l3['Close'].shift(1)).dropna()
sigma_l3 = float(log_r_l3.std()) * np.sqrt(252)

# Parámetros del contrato
K_l3, T_l3, r_l3 = S0_l3 * 1.05, 1.0, 0.03
N_l3 = 100_000

print(f'Activo: {ticker_l3} | S₀ = {S0_l3:.2f} USD | σ_b = {sigma_l3:.4f}')
print(f'K = {K_l3:.2f} (5% OTM) | T = {T_l3} año | r = {r_l3:.2%}')

In [ ]:
# ── Laboratorio 3: Simulación y comparación con Black-Scholes ─────────────

np.random.seed(0)
Z_l3  = np.random.standard_normal(N_l3)
ST_l3 = S0_l3 * np.exp((r_l3 - 0.5*sigma_l3**2)*T_l3 + sigma_l3*np.sqrt(T_l3)*Z_l3)
payoffs_l3 = np.maximum(ST_l3 - K_l3, 0)

mc_price_l3 = np.exp(-r_l3*T_l3) * payoffs_l3.mean()
se_l3       = np.exp(-r_l3*T_l3) * payoffs_l3.std() / np.sqrt(N_l3)
ci_lo_l3    = mc_price_l3 - 1.96*se_l3
ci_hi_l3    = mc_price_l3 + 1.96*se_l3

d1_l3 = (np.log(S0_l3/K_l3) + (r_l3+0.5*sigma_l3**2)*T_l3) / (sigma_l3*np.sqrt(T_l3))
d2_l3 = d1_l3 - sigma_l3*np.sqrt(T_l3)
bs_l3 = S0_l3*norm.cdf(d1_l3) - K_l3*np.exp(-r_l3*T_l3)*norm.cdf(d2_l3)

# Gráfico de la distribución de ST
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(ST_l3, bins=80, color='steelblue', edgecolor='white',
             alpha=0.8, density=True)
axes[0].axvline(K_l3, color='red', ls='--', lw=2, label=f'K = {K_l3:.0f}')
axes[0].axvline(S0_l3, color='green', ls='--', lw=2, label=f'S₀ = {S0_l3:.0f}')
axes[0].set_title('Distribución de $S_T$ (bajo medida $Q$)')
axes[0].set_xlabel('$S_T$ (USD)'); axes[0].legend()

axes[1].hist(payoffs_l3[payoffs_l3>0], bins=60, color='coral',
             edgecolor='white', alpha=0.8, density=True)
axes[1].axvline(mc_price_l3*np.exp(r_l3*T_l3), color='navy', ls='--',
                lw=2, label=f'Payoff medio = {payoffs_l3.mean():.2f}')
axes[1].set_title('Distribución de Payoffs > 0')
axes[1].set_xlabel('Payoff (USD)'); axes[1].legend()

plt.suptitle('Lab 3: Valuación Monte Carlo — Call Europeo SPY', fontweight='bold')
plt.tight_layout(); plt.show()

print('\n' + '='*55)
print(f'  Precio Monte Carlo:   {mc_price_l3:10.4f} USD')
print(f'  Error Estándar:       {se_l3:10.6f} USD')
print(f'  IC 95%:               [{ci_lo_l3:.4f}, {ci_hi_l3:.4f}]')
print(f'  Precio Black-Scholes: {bs_l3:10.4f} USD')
print(f'  Diferencia MC - BS:   {mc_price_l3-bs_l3:10.6f} USD')
print('='*55)

### Conclusiones — Laboratorio 3

El precio Monte Carlo converge al de Black-Scholes. La diferencia residual es atribuible a la aleatoriedad de la simulación y tiende a cero cuando $N\to\infty$. El error estándar decrece como $1/\sqrt{N}$. Monte Carlo es especialmente valioso para opciones exóticas donde no existen soluciones analíticas cerradas.

---
# Laboratorio 4 — Opción Americana con Longstaff–Schwartz

**Objetivo:** Valuar una *put* americana sobre SPY con el algoritmo de Longstaff-Schwartz  
y cuantificar la prima de americanidad.

**Idea clave:** En cada tiempo $t$ y para trayectorias *in-the-money*, se estima el valor de  
continuación $C(t, S_t)$ mediante regresión OLS con funciones base $\{1, S_t, S_t^2\}$.  
Se ejerce si $\max(K-S_t,0) > \hat{C}(t,S_t)$.

In [ ]:
# ── Laboratorio 4: Calibración ────────────────────────────────────────────

ticker_l4 = 'SPY'
data_l4   = yf.download(ticker_l4, start='2023-01-01', end='2024-01-01', progress=False)
if isinstance(data_l4.columns, pd.MultiIndex):
    data_l4.columns = data_l4.columns.droplevel(1)

S0_l4    = float(data_l4['Close'].iloc[-1])
log_r_l4 = np.log(data_l4['Close']/data_l4['Close'].shift(1)).dropna()
sigma_l4 = float(log_r_l4.std()) * np.sqrt(252)

K_l4, T_l4, r_l4 = S0_l4*0.95, 1.0, 0.05
M_l4, n_l4 = 10_000, 100
dt_l4 = T_l4 / n_l4

print(f'Activo: {ticker_l4} | S₀ = {S0_l4:.2f} | σ_b = {sigma_l4:.4f}')
print(f'K = {K_l4:.2f} (5% ITM put) | T = 1 año | r = {r_l4:.2%}')
print(f'Trayectorias M = {M_l4} | Pasos n = {n_l4}')

In [ ]:
# ── Laboratorio 4: Simulación y Longstaff-Schwartz ────────────────────────

np.random.seed(42)
paths_l4 = np.zeros((n_l4+1, M_l4));  paths_l4[0] = S0_l4
for t in range(1, n_l4+1):
    Z = np.random.normal(0, 1, M_l4)
    paths_l4[t] = paths_l4[t-1] * np.exp(
        (r_l4 - 0.5*sigma_l4**2)*dt_l4 + sigma_l4*np.sqrt(dt_l4)*Z)

# Longstaff-Schwartz backward induction
CF_l4   = np.maximum(K_l4 - paths_l4[-1], 0)
tau_l4  = np.full(M_l4, n_l4)

for t in range(n_l4-1, 0, -1):
    itm = np.where(K_l4 - paths_l4[t] > 0)[0]
    if len(itm) > 0:
        S_itm = paths_l4[t][itm]
        y = CF_l4[itm] * np.exp(-r_l4*dt_l4*(tau_l4[itm]-t))
        coef   = np.polyfit(S_itm, y, 2)
        C_hat  = np.polyval(coef, S_itm)
        iv     = np.maximum(K_l4 - S_itm, 0)
        ex_idx = itm[iv > C_hat]
        CF_l4[ex_idx]  = iv[iv > C_hat]
        tau_l4[ex_idx] = t

V_am_l4 = np.mean(CF_l4 * np.exp(-r_l4*dt_l4*tau_l4))

# Black-Scholes europeo comparable
d1_l4 = (np.log(S0_l4/K_l4) + (r_l4+0.5*sigma_l4**2)*T_l4)/(sigma_l4*np.sqrt(T_l4))
d2_l4 = d1_l4 - sigma_l4*np.sqrt(T_l4)
V_eu_l4 = K_l4*np.exp(-r_l4*T_l4)*norm.cdf(-d2_l4) - S0_l4*norm.cdf(-d1_l4)

# Visualización de trayectorias y distribución de ejercicio
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i in range(min(50, M_l4)):
    axes[0].plot(paths_l4[:, i], alpha=0.15, lw=0.8, color='steelblue')
axes[0].axhline(K_l4, color='red', ls='--', lw=2, label=f'K = {K_l4:.0f}')
axes[0].set_title('Trayectorias simuladas (GBM)')
axes[0].set_xlabel('Pasos de tiempo'); axes[0].set_ylabel('Precio (USD)')
axes[0].legend()

axes[1].hist(tau_l4, bins=30, color='coral', edgecolor='white', alpha=0.85)
axes[1].axvline(n_l4, color='navy', ls='--', lw=2, label='Vencimiento')
axes[1].set_title('Distribución del tiempo de ejercicio óptimo')
axes[1].set_xlabel('Paso de ejercicio'); axes[1].legend()

plt.suptitle('Lab 4: Longstaff-Schwartz — Put Americana SPY', fontweight='bold')
plt.tight_layout(); plt.show()

print('\n' + '='*50)
print(f'  Opción Americana (L-S):    {V_am_l4:.4f} USD')
print(f'  Opción Europea  (B-S):     {V_eu_l4:.4f} USD')
print(f'  Prima de americanidad:     {V_am_l4-V_eu_l4:.4f} USD')
print('='*50)

### Conclusiones — Laboratorio 4

El algoritmo de Longstaff-Schwartz captura la prima de americanidad (valor de la flexibilidad de ejercicio temprano). La regresión OLS sobre trayectorias *in-the-money* aproxima eficientemente el valor de continuación. El resultado es sensible a $M$ (número de trayectorias) y al grado del polinomio de regresión; con $M=10{,}000$ y grado 2, la convergencia es adecuada.

---
# Laboratorio 5 — Opción Parisina y Reloj de Barrera

**Objetivo:** Valuar una opción Parisina *Down-and-Out Put* sobre SPY implementando  
el "reloj de barrera" y analizar la sensibilidad a la duración $D$.

**Concepto clave — Memoria temporal:**  
La barrera se activa solo si el precio permanece $D$ días **consecutivos** bajo $H$.  
El contador se reinicia a cero cuando el precio regresa por encima de $H$.

$$\text{Precio} = e^{-rT}\,\frac{1}{N}\sum_{i: \text{no knocked}} \max(K-S_T^{(i)},0)$$

In [ ]:
# ── Laboratorio 5: Calibración ────────────────────────────────────────────

ticker_l5 = 'SPY'
data_l5   = yf.download(ticker_l5, start='2020-01-01', end='2023-01-01', progress=False)
if isinstance(data_l5.columns, pd.MultiIndex):
    data_l5.columns = data_l5.columns.droplevel(1)
prices_l5 = data_l5['Close']

S0_l5    = float(prices_l5.iloc[-1])
log_r_l5 = np.log(prices_l5/prices_l5.shift(1)).dropna()
sigma_l5 = float(log_r_l5.std()) * np.sqrt(252)

K_l5, T_l5, r_l5, H_l5 = S0_l5*0.95, 1.0, 0.02, S0_l5*0.90
N_l5, n_l5 = 100_000, 252
dt_l5 = T_l5 / n_l5
D_vals = [5, 10, 20, 30]

print(f'Activo: {ticker_l5} | S₀ = {S0_l5:.2f} | σ_b = {sigma_l5:.4f}')
print(f'H = {H_l5:.2f} (10% bajo S₀) | K = {K_l5:.2f} | D ∈ {D_vals}')

In [ ]:
# ── Laboratorio 5: Simulación de trayectorias (una vez para todos los D) ──

np.random.seed(42)
paths_l5 = np.zeros((N_l5, n_l5+1));  paths_l5[:, 0] = S0_l5
for t in range(n_l5):
    z = np.random.normal(0, 1, N_l5)
    paths_l5[:, t+1] = paths_l5[:, t] * np.exp(
        (r_l5 - 0.5*sigma_l5**2)*dt_l5 + sigma_l5*np.sqrt(dt_l5)*z)

print(f'Simuladas {N_l5:,} trayectorias × {n_l5+1} pasos.')

In [ ]:
# ── Laboratorio 5: Reloj de barrera y análisis de sensibilidad ────────────

def parisian_put_price(D, paths, K, T, r, H, N, n):
    """Valuación Monte Carlo de una Put Down-and-Out Parisina.
    
    Parámetros
    ----------
    D      : días consecutivos necesarios bajo H para activar el knock-out
    paths  : array (N, n+1) de trayectorias de precios simuladas
    K, T, r: precio de ejercicio, vencimiento, tasa libre de riesgo
    H      : nivel de barrera
    N, n   : número de trayectorias y pasos
    
    Retorna
    -------
    (precio, error_estandar)
    """
    payoffs = np.zeros(N)
    for i in range(N):
        consec, knocked = 0, False
        for t in range(1, n+1):
            if paths[i, t] < H:
                consec += 1
                if consec >= D:
                    knocked = True
                    break
            else:
                consec = 0
        if not knocked:
            payoffs[i] = max(0.0, K - paths[i, -1])

    disc   = np.exp(-r * T)
    price  = disc * payoffs.mean()
    se     = disc * payoffs.std() / np.sqrt(N)
    return price, se


# Evaluación para cada D
prices_par, errors_par = [], []
print(f'{"D (días)":>10} {"Precio":>12} {"Error Std":>12}')
print('-' * 38)
for D in D_vals:
    p, e = parisian_put_price(D, paths_l5, K_l5, T_l5, r_l5, H_l5, N_l5, n_l5)
    prices_par.append(p); errors_par.append(e)
    print(f'{D:>10}d {p:>12.4f} {e:>12.4f}')

In [ ]:
# ── Laboratorio 5: Gráfico de sensibilidad ────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel izquierdo: precio vs D con barras de error
axes[0].errorbar(D_vals, prices_par, yerr=errors_par, fmt='-o',
                 capsize=6, color='steelblue', ecolor='lightcoral',
                 markerfacecolor='navy', markersize=8, lw=2)
axes[0].set_title('Precio Parisiana vs. Duración D')
axes[0].set_xlabel('D (días consecutivos bajo H)')
axes[0].set_ylabel('Precio de la Opción (USD)')
axes[0].grid(True, ls='--', alpha=0.6)

# Panel derecho: ejemplo de trayectoria con reloj
sample_path = paths_l5[0, :]
steps = np.arange(len(sample_path))
axes[1].plot(steps, sample_path, color='steelblue', lw=1.2, label='Trayectoria')
axes[1].axhline(H_l5, color='red', ls='--', lw=2, label=f'Barrera H={H_l5:.0f}')
axes[1].axhline(K_l5, color='green', ls=':', lw=1.5, label=f'K={K_l5:.0f}')
axes[1].fill_between(steps, sample_path, H_l5,
                     where=sample_path < H_l5,
                     color='red', alpha=0.2, label='Zona de riesgo')
axes[1].set_title('Trayectoria ilustrativa y zona de barrera')
axes[1].set_xlabel('Paso de tiempo'); axes[1].legend(fontsize=9)

plt.suptitle('Lab 5: Opción Parisina Down-and-Out Put — Reloj de Barrera',
             fontweight='bold')
plt.tight_layout(); plt.show()

### Conclusiones — Laboratorio 5

**Memoria temporal vs. barrera estándar:**  
La opción parisiana es más cara que la barrera estándar equivalente porque la probabilidad de  
knock-out es menor: se requieren $D$ días *consecutivos* bajo $H$, no un solo toque.

**Monotonía respecto a $D$:**  
$$\text{Precio}(D_1) < \text{Precio}(D_2) \quad \text{si} \quad D_1 < D_2$$  
Para $D\to\infty$, la opción converge a una put europea sin barrera.

**Implicación matemática:**  
El «reloj de barrera» introduce una dimensión temporal en la condición de desactivación,  
modela regulaciones o umbrales que solo se activan tras una permanencia sostenida bajo un umbral,  
y ofrece un marco más matizado para la gestión de riesgos y la estructuración de productos financieros.

---

## Conclusión General

La progresión de los siete módulos de este compilado sigue una lógica de construcción incremental:

| Módulo | Aportación principal |
|--------|---------------------|
| RA1000 | Los factores CAPM y FF3 explican sistemáticamente los retornos de AAPL |
| Tarea | Andamiaje teórico: Poisson, Markov, martingalas |
| Lab 1 | GBM como primer modelo continuo; sus limitaciones motivan lo siguiente |
| Lab 2 | La volatilidad no es constante — clustering es un hecho empírico robusto |
| Lab 3 | Monte Carlo bajo $Q$ converge a Black-Scholes |
| Lab 4 | Longstaff-Schwartz extiende MC a ejercicio temprano |
| Lab 5 | Las opciones exóticas (parisinas) introducen memoria temporal |

La combinación de teoría estocástica rigurosa con implementación computacional en Python ofrece una base sólida para la gestión de riesgos y la valuación de derivados en los mercados financieros modernos.